In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import * 
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
import pandas as pd
import os

# Confirm no SPARK_HOME leaking in
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))  # Must print None, if the variable is set in system variables, delete it, since we will be using the spark config from venv.

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]


spark = SparkSession.builder \
    .appName("DeltaLocal_Test") \
    .master("local[*]") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.1.0") \
    .config("spark.driver.extraJavaOptions", "-Divy.home=E:/Media_Intelligence_Project/ivy2_cache") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark version:", spark.version)

SPARK_HOME: None
Spark version: 4.1.1


In [2]:
assignment4_df = spark.read.format('csv').option('header',True).load('C:\\Users\\My pc\\Downloads\\Assignment4_data.csv')
assignment4_df = assignment4_df.withColumn('Date',to_date('Date','dd-MM-yyyy'))
assignment4_df.write.format('delta').mode('overwrite').save("C:\\Users\\My pc\\Downloads\\Telegram Desktop\\Sample_Table")

In [3]:
df_delta = spark.read.format('delta') \
    .load('C:\\Users\\My pc\\Downloads\\Telegram Desktop\\Sample_Table')
df_delta.show(5)

+--------+--------------------+--------------------+---------+--------------------+----+----------+----------+
|Platform|             User_ID|      Transaction_ID|  Item_ID|           Item_Name|Cost| Timestamp|      Date|
+--------+--------------------+--------------------+---------+--------------------+----+----------+----------+
|    5687|jRuyDt3R95noXaaCC...| 2618916649179830000|496171947|outfit_flecktarn_...| 600|1542161258|2018-11-14|
|    5689|GXawGFPgJT6ZwhnwQ...|  249778375071398000|496171947|outfit_flecktarn_...| 600|1544546862|2018-12-11|
|    5687|Klx3Xrr5KFRZYURCh...| 5583394722751080000|496171947|outfit_flecktarn_...| 600|1543027797|2018-11-24|
|    5687|UYdBq3ZRyT3tTVj_z...|11263112910322700000|496171947|outfit_flecktarn_...| 600|1542140502|2018-11-13|
|    5688|2OfIAVL2_67qhmD1c...| 6289619434745350000|496171947|outfit_flecktarn_...| 600|1543648557|2018-12-01|
+--------+--------------------+--------------------+---------+--------------------+----+----------+----------+
o